In [1]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Scenario-1: Global Nearest by Friend Distance -> Leader by MIN SUM DISTANCE
# NO FALLBACK / NO PENALTY / NO NORMALIZATION
#
# IDEA (Doctor's direction):
# 1) For each voter u in Interest Community I:
#       choose nearest node globally (can be outside I)
#       => MIN friend distance dist = 1 - jaccard_sim
#
# 2) Candidate set C = unique chosen neighbors from step (1)
#
# 3) For each candidate c in C:
#       sum_dist(c) = sum of distances from c to all nodes in I
#                     using ONLY available distances (no penalty for missing)
#       coverage(c) = how many nodes in I have available distance to c
#
# 4) Winner per interest community:
#       MIN sum_dist
#       tie-break: coverage DESC
#       tie-break: candidate ASC
#
# 5) Record decision basis:
#       sum_distance / tie_coverage / tie_id / no_nominee
#
# Saves:
#   - nominations per community
#   - candidate scores per community
#   - final summary
#   - optional distance edgelist
#   - optional within-I distance matrices
# =====================================================================================

# =========================
# Cell 0 — Config + Imports
# =========================
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict

# ----------------------------
# CONFIG
# ----------------------------
OP_ROOT = r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\2 التصويت\OP"
RUN_ID  = datetime.now().strftime("RUN_%Y%m%d_%H%M%S")

project_dir = os.path.join(OP_ROOT, RUN_ID)
outputs_dir = os.path.join(project_dir, "outputs")
os.makedirs(outputs_dir, exist_ok=True)

# OPTIONAL (not used in current logic)
friend_xlsx = r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\1 uu_comm_dete\Douban_uu_membership_with_friends_New.xlsx"

# Friends similarity EDGE LIST
friends_sim_csv = r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\Friend_Friend_Sim_Matrix\friends_jaccard_sim_nonzero.csv"

# Interest community files
interest_files = [
    r"D:\PHD\results\Douban User Movie Ratinh Pred Comm\User Comm above 14 V2\003_comm_8_U104_I3510.txt",
    r"D:\PHD\results\Douban User Movie Ratinh Pred Comm\User Comm above 14 V2\006_comm_14_U137_I3673.txt",
    r"D:\PHD\results\Douban User Movie Ratinh Pred Comm\User Comm above 14 V2\023_comm_84_U117_I3855.txt",
    r"D:\PHD\results\Douban User Movie Ratinh Pred Comm\User Comm above 14 V2\026_comm_96_U137_I4033.txt",
    r"D:\PHD\results\Douban User Movie Ratinh Pred Comm\User Comm above 14 V2\028_comm_101_U80_I3034.txt",
    r"D:\PHD\results\Douban User Movie Ratinh Pred Comm\User Comm above 14 V2\045_comm_287_U118_I2606.txt",
]

# Performance
CHUNKSIZE = 400_000
EPS = 1e-12

# Output switches
WRITE_DISTANCE_EDGELIST = True
WRITE_WITHIN_I_MATRICES = True
WRITE_NOMINATION_FILES  = True
WRITE_CANDIDATE_FILES   = True

print("[INFO] RUN_ID:", RUN_ID)
print("[INFO] project_dir:", project_dir)
print("[INFO] outputs_dir:", outputs_dir)
print("[INFO] friends_sim_csv:", friends_sim_csv)
print("[INFO] CHUNKSIZE:", CHUNKSIZE)

# =========================
# Cell 1 — Helpers + Checks
# =========================
def safe_strip_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.astype(str).str.strip()
    return df

def ensure_path_exists(p: str, name: str = "") -> str:
    if os.path.exists(p):
        return p
    raise FileNotFoundError(f"[ERROR] Missing {name}: {p}")

def parse_interest_comm_id(path: str) -> int:
    m = re.search(r"comm_(\d+)", os.path.basename(path))
    if not m:
        raise ValueError(f"Cannot parse comm_id from filename: {path}")
    return int(m.group(1))

friends_sim_csv = ensure_path_exists(friends_sim_csv, "friends_sim_csv")
for fp in interest_files:
    ensure_path_exists(fp, f"interest_file: {os.path.basename(fp)}")

# Output folders
step1_dir   = os.path.join(outputs_dir, "step1_interest")
vote_dir    = os.path.join(outputs_dir, "scenario1_global_friend_distance_sumdist_no_fallback")
nom_dir     = os.path.join(vote_dir, "nominations_per_interest")
cand_dir    = os.path.join(vote_dir, "candidate_scores_per_interest")
final_dir   = os.path.join(vote_dir, "final")
mat_dir     = os.path.join(vote_dir, "friend_distance_matrices_within_I")

for d in [step1_dir, vote_dir, nom_dir, cand_dir, final_dir]:
    os.makedirs(d, exist_ok=True)
if WRITE_WITHIN_I_MATRICES:
    os.makedirs(mat_dir, exist_ok=True)

print("[OK] Input paths verified. Output folders created.")

# ==========================================
# Cell 2 — STEP-1 Interest Extract
# ==========================================
interest_rows = []
files_meta = []

for fp in interest_files:
    ic = parse_interest_comm_id(fp)
    dfI = safe_strip_cols(pd.read_csv(fp, sep=r"\s+", engine="python"))

    if not {"User", "Movie", "Rating"}.issubset(set(dfI.columns)):
        raise ValueError(f"[ERROR] Interest file must contain User Movie Rating: {fp}")

    users = sorted(dfI["User"].astype(int).unique().tolist())
    for u in users:
        interest_rows.append({"interest_comm_id": ic, "User": int(u)})

    files_meta.append({
        "interest_comm_id": ic,
        "file_path": fp,
        "rows": int(len(dfI)),
        "n_users": int(len(users)),
        "n_items": int(dfI["Movie"].nunique())
    })

interest_membership = pd.DataFrame(interest_rows).sort_values(["interest_comm_id", "User"]).reset_index(drop=True)
interest_sizes = (
    interest_membership.groupby("interest_comm_id")["User"].nunique()
    .reset_index(name="interest_comm_size")
    .sort_values("interest_comm_id").reset_index(drop=True)
)
df_files_meta = pd.DataFrame(files_meta).sort_values("interest_comm_id").reset_index(drop=True)

interest_membership.to_csv(os.path.join(step1_dir, "interest_membership.csv"), index=False, encoding="utf-8-sig")
interest_sizes.to_csv(os.path.join(step1_dir, "interest_community_sizes.csv"), index=False, encoding="utf-8-sig")
df_files_meta.to_csv(os.path.join(step1_dir, "interest_files_meta.csv"), index=False, encoding="utf-8-sig")

interest_groups = (
    interest_membership.groupby("interest_comm_id")["User"]
    .apply(lambda x: sorted(x.astype(int).tolist()))
    .to_dict()
)

all_users = set()
for ic, users in interest_groups.items():
    all_users.update(users)

print("[OK] Interest communities:", len(interest_groups))
print("[OK] Total unique interest users:", len(all_users))
print(interest_sizes.to_string(index=False))

# ==========================================
# Cell 3 — One-pass scan friends edge list
#          Build nearest global neighbor for each interest user
#          + store distances from interest users to ANY seen node
# ==========================================
hdr = safe_strip_cols(pd.read_csv(friends_sim_csv, nrows=0))
need_cols = {"friend_1", "friend_2", "jaccard_sim"}
if not need_cols.issubset(set(hdr.columns)):
    raise ValueError(f"[ERROR] friends_sim_csv must contain {need_cols}, found {set(hdr.columns)}")

usecols = ["friend_1", "friend_2", "jaccard_sim"]

# Best global nearest neighbor for each interest user
best_nb   = {int(u): None for u in all_users}
best_sim  = {int(u): -1.0 for u in all_users}
best_dist = {int(u): None for u in all_users}

# Presence in edge list
present_users = set()

# Distance lookup:
# adj_dist[u][v] = distance between interest user u and node v
# We store only when u is in all_users
adj_dist = defaultdict(dict)

dist_path = os.path.join(vote_dir, "friends_distance_edgelist.csv")
dist_writer = None
if WRITE_DISTANCE_EDGELIST:
    dist_writer = open(dist_path, "w", encoding="utf-8")
    dist_writer.write("friend_1,friend_2,jaccard_sim,jaccard_dist\n")

for chunk in pd.read_csv(friends_sim_csv, chunksize=CHUNKSIZE, usecols=usecols):
    chunk = safe_strip_cols(chunk)

    if WRITE_DISTANCE_EDGELIST:
        tmp = chunk.copy()
        tmp["jaccard_dist"] = 1.0 - tmp["jaccard_sim"].astype(float)
        tmp.to_csv(dist_writer, index=False, header=False)

    a = chunk["friend_1"].astype(np.int64).to_numpy()
    b = chunk["friend_2"].astype(np.int64).to_numpy()
    w = chunk["jaccard_sim"].astype(float).to_numpy()

    for u, v, ww in zip(a, b, w):
        u = int(u)
        v = int(v)
        ww = float(ww)
        if u == v:
            continue

        d = 1.0 - ww

        # If u is an interest user -> store distance to v
        if u in best_nb:
            present_users.add(u)

            prev = adj_dist[u].get(v, None)
            if (prev is None) or (d < prev):
                adj_dist[u][v] = d

            curw = best_sim[u]
            curnb = best_nb[u]
            if (ww > curw) or (abs(ww - curw) <= EPS and (curnb is None or v < curnb)):
                best_sim[u] = ww
                best_nb[u] = v
                best_dist[u] = d

        # If v is an interest user -> store distance to u
        if v in best_nb:
            present_users.add(v)

            prev = adj_dist[v].get(u, None)
            if (prev is None) or (d < prev):
                adj_dist[v][u] = d

            curw = best_sim[v]
            curnb = best_nb[v]
            if (ww > curw) or (abs(ww - curw) <= EPS and (curnb is None or u < curnb)):
                best_sim[v] = ww
                best_nb[v] = u
                best_dist[v] = d

if dist_writer:
    dist_writer.close()

print("[OK] Friends scan complete.")
print("[INFO] Interest users present in friends edge list:", len(present_users))
print("[INFO] Interest users missing (ABSTAIN):", len(all_users) - len(present_users))

# ==========================================
# Cell 4 — Nomination + Candidate Scoring + Election
# ==========================================
summary_rows = []

for ic in sorted(interest_groups.keys()):
    I = [int(u) for u in interest_groups[ic]]
    Iset = set(I)
    total = len(I)

    # -------------------------
    # Step A: Nominations
    # -------------------------
    nomination_rows = []
    abstain_count = 0

    for u in I:
        v = best_nb[u]

        if v is None:
            abstain_count += 1
            nomination_rows.append({
                "interest_comm_id": int(ic),
                "voter": int(u),
                "chosen_candidate": None,
                "chosen_sim": None,
                "chosen_dist": None,
                "abstained": True
            })
        else:
            nomination_rows.append({
                "interest_comm_id": int(ic),
                "voter": int(u),
                "chosen_candidate": int(v),
                "chosen_sim": float(best_sim[u]),
                "chosen_dist": float(best_dist[u]),
                "abstained": False
            })

    df_nom = pd.DataFrame(nomination_rows)

    if WRITE_NOMINATION_FILES:
        df_nom.to_csv(
            os.path.join(nom_dir, f"nominations_global_friend_comm_{int(ic)}.csv"),
            index=False,
            encoding="utf-8-sig"
        )
        df_nom.to_excel(
            os.path.join(nom_dir, f"nominations_global_friend_comm_{int(ic)}.xlsx"),
            index=False
        )

    # -------------------------
    # Step B: Candidate set
    # -------------------------
    df_nom_valid = df_nom[df_nom["abstained"] == False].copy()

    if len(df_nom_valid) == 0:
        # All abstain / no nominee
        summary_rows.append({
            "interest_comm_id": int(ic),
            "interest_size": int(total),
            "nominating_voters": 0,
            "abstain_count": int(abstain_count),
            "nomination_coverage": 0.0,
            "unique_nominees": 0,
            "leader": None,
            "leader_sum_dist": None,
            "leader_coverage_count": 0,
            "leader_coverage_ratio": 0.0,
            "leader_nominated_by_count": 0,
            "leader_in_I": None,
            "decision_basis": "no_nominee",
            "tie_group_size": 0,
            "runner_up": None,
            "runner_up_sum_dist": None,
            "runner_up_coverage_count": None,
            "sum_margin": None,
            "coverage_margin": None
        })
        continue

    candidates = sorted(df_nom_valid["chosen_candidate"].astype(int).unique().tolist())

    nominated_by_count_map = (
        df_nom_valid.groupby("chosen_candidate")["voter"]
        .count()
        .astype(int)
        .to_dict()
    )

    # -------------------------
    # Step C: Score each candidate
    # -------------------------
    cand_rows = []

    for c in candidates:
        sum_dist = 0.0
        coverage_count = 0
        missing_count = 0

        for u in I:
            if u == c:
                # if candidate itself is inside I, self-distance is known = 0
                d = 0.0
                coverage_count += 1
                sum_dist += d
            else:
                d = adj_dist.get(u, {}).get(c, None)
                if d is None:
                    missing_count += 1
                else:
                    coverage_count += 1
                    sum_dist += float(d)

        coverage_ratio = coverage_count / total if total else 0.0
        mean_dist_diag = (sum_dist / coverage_count) if coverage_count > 0 else None

        cand_rows.append({
            "interest_comm_id": int(ic),
            "candidate_id": int(c),
            "sum_dist": float(sum_dist),
            "coverage_count": int(coverage_count),
            "coverage_ratio": float(coverage_ratio),
            "missing_count": int(missing_count),
            "mean_dist_diag": mean_dist_diag,   # diagnostic only
            "nominated_by_count": int(nominated_by_count_map.get(c, 0)),
            "candidate_in_I": bool(c in Iset)
        })

    df_cand = pd.DataFrame(cand_rows)

    if WRITE_CANDIDATE_FILES:
        df_cand.to_csv(
            os.path.join(cand_dir, f"candidate_scores_comm_{int(ic)}.csv"),
            index=False,
            encoding="utf-8-sig"
        )
        df_cand.to_excel(
            os.path.join(cand_dir, f"candidate_scores_comm_{int(ic)}.xlsx"),
            index=False
        )

    # -------------------------
    # Step D: Determine winner
    # Rule:
    #   1) MIN sum_dist
    #   2) coverage_count DESC
    #   3) candidate_id ASC
    # -------------------------
    df_rank = df_cand.sort_values(
        by=["sum_dist", "coverage_count", "candidate_id"],
        ascending=[True, False, True]
    ).reset_index(drop=True)

    leader_row = df_rank.iloc[0].copy()
    leader = int(leader_row["candidate_id"])
    leader_sum_dist = float(leader_row["sum_dist"])
    leader_coverage_count = int(leader_row["coverage_count"])
    leader_coverage_ratio = float(leader_row["coverage_ratio"])
    leader_nominated_by_count = int(leader_row["nominated_by_count"])
    leader_in_I = bool(leader_row["candidate_in_I"])

    # Decision basis
    min_sum = df_rank["sum_dist"].min()
    df_sum_tied = df_rank[np.abs(df_rank["sum_dist"] - min_sum) <= EPS].copy()

    if len(df_sum_tied) == 1:
        decision_basis = "sum_distance"
        tie_group_size = 1
    else:
        max_cov = df_sum_tied["coverage_count"].max()
        df_cov_tied = df_sum_tied[df_sum_tied["coverage_count"] == max_cov].copy()

        if len(df_cov_tied) == 1:
            decision_basis = "tie_coverage"
            tie_group_size = len(df_sum_tied)
        else:
            decision_basis = "tie_id"
            tie_group_size = len(df_cov_tied)

    # Runner-up diagnostics
    if len(df_rank) >= 2:
        runner_row = df_rank.iloc[1].copy()
        runner_up = int(runner_row["candidate_id"])
        runner_up_sum_dist = float(runner_row["sum_dist"])
        runner_up_coverage_count = int(runner_row["coverage_count"])
        sum_margin = float(runner_up_sum_dist - leader_sum_dist)
        coverage_margin = int(leader_coverage_count - runner_up_coverage_count)
    else:
        runner_up = None
        runner_up_sum_dist = None
        runner_up_coverage_count = None
        sum_margin = None
        coverage_margin = None

    nominating_voters = total - abstain_count
    nomination_coverage = (nominating_voters / total) if total else 0.0

    summary_rows.append({
        "interest_comm_id": int(ic),
        "interest_size": int(total),
        "nominating_voters": int(nominating_voters),
        "abstain_count": int(abstain_count),
        "nomination_coverage": float(nomination_coverage),
        "unique_nominees": int(len(candidates)),
        "leader": int(leader),
        "leader_sum_dist": float(leader_sum_dist),
        "leader_coverage_count": int(leader_coverage_count),
        "leader_coverage_ratio": float(leader_coverage_ratio),
        "leader_nominated_by_count": int(leader_nominated_by_count),
        "leader_in_I": bool(leader_in_I),
        "decision_basis": decision_basis,
        "tie_group_size": int(tie_group_size),
        "runner_up": runner_up,
        "runner_up_sum_dist": runner_up_sum_dist,
        "runner_up_coverage_count": runner_up_coverage_count,
        "sum_margin": sum_margin,
        "coverage_margin": coverage_margin
    })

    # -------------------------
    # Optional: within-I distance matrix
    # -------------------------
    if WRITE_WITHIN_I_MATRICES:
        mat = pd.DataFrame(np.nan, index=I, columns=I, dtype=float)
        np.fill_diagonal(mat.values, 0.0)

        for u in I:
            for v in I:
                if u == v:
                    mat.at[u, v] = 0.0
                else:
                    d = adj_dist.get(u, {}).get(v, None)
                    if d is not None:
                        mat.at[u, v] = float(d)

        mat.to_excel(
            os.path.join(mat_dir, f"friend_distance_within_I_comm_{int(ic)}.xlsx"),
            index=True
        )

# ==========================================
# Cell 5 — Final Summary
# ==========================================
df_summary = pd.DataFrame(summary_rows).sort_values("interest_comm_id").reset_index(drop=True)

df_summary.to_csv(
    os.path.join(final_dir, "scenario1_global_friend_sumdist_no_fallback_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)
df_summary.to_excel(
    os.path.join(final_dir, "scenario1_global_friend_sumdist_no_fallback_summary.xlsx"),
    index=False
)

print("\n[DONE] Scenario-1 (Leader by MIN SUM DISTANCE, No Fallback) finished.")
print("[PATH] vote_dir   :", vote_dir)
print("[PATH] nom_dir    :", nom_dir)
print("[PATH] cand_dir   :", cand_dir)
print("[PATH] final_dir  :", final_dir)
if WRITE_DISTANCE_EDGELIST:
    print("[PATH] distance_edgelist:", dist_path)
if WRITE_WITHIN_I_MATRICES:
    print("[PATH] matrices_dir:", mat_dir)

print("\n[SUMMARY]")
print(df_summary.to_string(index=False))

[INFO] RUN_ID: RUN_20260309_031845
[INFO] project_dir: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\2 التصويت\OP\RUN_20260309_031845
[INFO] outputs_dir: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\2 التصويت\OP\RUN_20260309_031845\outputs
[INFO] friends_sim_csv: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\Douban Reco\Friend_Friend_Sim_Matrix\friends_jaccard_sim_nonzero.csv
[INFO] CHUNKSIZE: 400000
[OK] Input paths verified. Output folders created.
[OK] Interest communities: 6
[OK] Total unique interest users: 693
 interest_comm_id  interest_comm_size
                8                 104
               14                 137
               84                 117
               96                 137
              101                  80
              287                 118
[OK] Friends scan complete.
[INFO] Interest users present in friends edge list: 135
[INFO] Interest users missing (ABSTAIN): 558

[DONE] Scenario-1 (Leader by MIN SUM DISTANCE, No Fallback) fin